In [1]:
import sys, torch
print(sys.version)        
print(torch.__version__)  

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
2.8.0+cu126


In [2]:
!pip install torch torchvision --quiet
!pip install torch-geometric --quiet
!pip install matplotlib seaborn scipy tqdm --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.7 MB/s eta 0:00:00a 0:00:01


In [3]:
import glob

# Find actual wheel locations
wheels = glob.glob('/kaggle/input/**/*.whl', recursive=True)
for w in wheels:
    print(w)

print(f"\nPython: {sys.version}")
print(f"Torch:  {torch.__version__}")

/kaggle/input/private-dataset/torch_spline_conv-1.2.2pt29cpu-cp312-cp312-linux_x86_64.whl
/kaggle/input/private-dataset/torch_sparse-0.6.18pt29cpu-cp312-cp312-linux_x86_64.whl
/kaggle/input/private-dataset/torch_cluster-1.6.3pt29cpu-cp312-cp312-linux_x86_64.whl
/kaggle/input/private-dataset/torch_scatter-2.1.2pt29cpu-cp312-cp312-linux_x86_64.whl
/kaggle/input/datasets/aaryaupi/cached-artificats/wheels/torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl
/kaggle/input/datasets/aaryaupi/cached-artificats/wheels/torch_spline_conv-1.2.2-cp312-cp312-linux_x86_64.whl
/kaggle/input/datasets/aaryaupi/cached-artificats/wheels/torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl
/kaggle/input/datasets/aaryaupi/cached-artificats/wheels/torch_cluster-1.6.3-cp312-cp312-linux_x86_64.whl

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch:  2.8.0+cu126


# Pytorch Cuda Wheels to access PyG scatter sparse 

In [4]:
import subprocess

# Detect PyTorch and CUDA versions
torch_version = torch.__version__.split('+')[0]  # "2.8.0"
cuda_version = torch.version.cuda.replace('.', '')  # "126" from "12.6"

print(f"PyTorch: {torch_version}, CUDA: {cuda_version}")

# Install PyG extensions for your exact versions
pyg_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html"

packages = [
    'torch-scatter',
    'torch-sparse', 
    'torch-cluster',
    'torch-spline-conv'
]

for pkg in packages:
    print(f"\nInstalling {pkg}...")
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, 
         '-f', pyg_url, '--no-cache-dir', '-q'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  ✓ {pkg} installed successfully")
    else:
        print(f"  ✗ {pkg} failed: {result.stderr[:300]}")

PyTorch: 2.8.0, CUDA: 126

Installing torch-scatter...
  ✓ torch-scatter installed successfully

Installing torch-sparse...
  ✓ torch-sparse installed successfully

Installing torch-cluster...
  ✓ torch-cluster installed successfully

Installing torch-spline-conv...
  ✓ torch-spline-conv installed successfully


In [5]:
from torch_geometric.transforms import SamplePoints, NormalizeScale, KNNGraph
import torch_geometric.transforms as T
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch, Data
from torch_geometric.nn import global_max_pool
from torch_cluster import knn_graph
from torch.cuda.amp import autocast, GradScaler
 
import os, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import trange, tqdm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from scipy.spatial import cKDTree
 
# ─────────────────────────────────────────────────────
# DEVICE / HYPERPARAMS
# ─────────────────────────────────────────────────────
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CACHE_PATH = '/kaggle/input/datasets/aaryaupi/cached-artificats/modelnet40_final.pt'
MOTIF_CACHE = '/kaggle/input/datasets/aaryaupi/gin-latest-pt-file/modelnet40_with_motifs.pt'
K_NEIGHBORS = 20
NUM_POINTS   = 1024
BATCH_SIZE   = 12  # Optimized for P100
EPOCHS       = 250
LR           = 1e-3
NUM_CLASSES  = 40
USE_AUGMENTATION = False  # Set to True to enable augmentation
 
# ─────────────────────────────────────────────────────
# 1. LOAD CACHED DATA
# ─────────────────────────────────────────────────────
print("Loading cached ModelNet40...")
import torch_geometric.data.data
torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
])
 
# Try to load motif cache first, fallback to base cache
if os.path.exists(MOTIF_CACHE):
    print(f"  Found motif cache: {MOTIF_CACHE}")
    cache = torch.load(MOTIF_CACHE, map_location='cpu', weights_only=False, mmap=True)
    train_list = cache['train']
    test_list = cache['test']
    CLASSES = cache['classes']
    print(f"  ✓ Loaded {len(train_list)} train + {len(test_list)} test samples")
    print(f"  ✓ Features already include motifs (shape: {train_list[0].x.shape})")
    MOTIFS_PRECOMPUTED = True
elif os.path.exists(CACHE_PATH):
    print(f"  Found base cache: {CACHE_PATH}")
    cache = torch.load(CACHE_PATH, map_location='cpu', weights_only=False, mmap=True)
    train_list = cache['train']
    test_list = cache['test']
    CLASSES = cache['classes']
    print(f"  ✓ Loaded {len(train_list)} train + {len(test_list)} test samples")
    print(f"  ⚠ Motifs not precomputed, will compute now...")
    MOTIFS_PRECOMPUTED = False
else:
    raise FileNotFoundError(f"Cache not found at {CACHE_PATH} or {MOTIF_CACHE}")

Loading cached ModelNet40...
  Found base cache: /kaggle/input/datasets/aaryaupi/cached-artificats/modelnet40_final.pt
  ✓ Loaded 9843 train + 2468 test samples
  ⚠ Motifs not precomputed, will compute now...


In [6]:
def compute_triangle_counts(edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
    """
    Count triangles per node using sparse A² trick
    
    WHY THIS WORKS:
      For adjacency matrix A, (A²)[i,j] counts paths of length 2
      from i to j (common neighbors).
      A triangle exists when edge (i,j) exists AND i,j share a neighbor:
          triangles(i) = sum_j  A[i,j] * (A²)[i,j]
      Each triangle counted twice → divide by 2
    
    Complexity: O(N·k²) via matmul, ~20ms per cloud at N=1024, k=20
    
    Args:
        edge_index: [2, E] directed edges
        num_nodes: int N
    
    Returns:
        [N, 1] normalized triangle count per node
    """
    N = num_nodes
    src = edge_index[0]
    dst = edge_index[1]
    
    # Build dense adjacency matrix A [N, N]
    A = torch.zeros(N, N, dtype=torch.float32)
    A[src, dst] = 1.0
    
    # A² = A @ A
    A2 = A @ A  # [N, N] - counts common neighbors
    
    # Triangle count per node
    counts = (A * A2).sum(dim=1)  # [N]
    counts = counts / 2.0  # Each triangle counted twice
    
    # Normalize to [0, 1]
    max_c = counts.max()
    if max_c > 0:
        counts = counts / max_c
    
    return counts.unsqueeze(1)  # [N, 1]
 
 
def add_motif_features(data_list: list, k: int = 20, desc: str = '') -> list:
    """
    Add triangle motif counts as node features
    
    For each Data:
      1. Build static KNN graph from positions
      2. Compute triangle counts
      3. Append as extra feature → x becomes [N, 7]
    
    Args:
        data_list: list of PyG Data objects
        k: number of neighbors
        desc: progress bar description
    
    Returns:
        data_list with motif features added
    """
    for data in tqdm(data_list, desc=f'Motif scores {desc}', leave=False):
        pos = data.pos.numpy()  # [N, 3]
        N = pos.shape[0]
        
        # Build geometric KNN using cKDTree
        tree = cKDTree(pos)
        _, nn_idx = tree.query(pos, k=k + 1)  # [N, k+1]
        nn_idx = nn_idx[:, 1:]  # Drop self → [N, k]
        
        src = np.repeat(np.arange(N), k)  # [N*k]
        dst = nn_idx.flatten()  # [N*k]
        
        edge_index_geo = torch.tensor(
            np.stack([src, dst], axis=0),
            dtype=torch.long
        )
        
        # Compute triangle counts
        tri_counts = compute_triangle_counts(edge_index_geo, N)  # [N, 1]
        
        # Append to node features: x was [N, 6], now [N, 7]
        data.x = torch.cat([data.x, tri_counts], dim=1)
        
        # Store geometric edge_index (for reference)
        data.edge_index = edge_index_geo
    
    return data_list
 
 
# ─────────────────────────────────────────────────────
# 3. FARTHEST POINT SAMPLING
# ─────────────────────────────────────────────────────
def farthest_point_sample_simple(points: torch.Tensor, n_samples: int) -> torch.Tensor:
    """
    Simple FPS - selects maximally spread out points
    
    Args:
        points: [N, 3] point cloud (any device)
        n_samples: number of points to sample
    
    Returns:
        indices: [n_samples] indices of sampled points
    """
    N = points.shape[0]
    device = points.device
    
    if n_samples >= N:
        return torch.arange(N, device=device)
    
    sampled_indices = []
    distances = torch.ones(N, device=device) * 1e10
    
    # Start with random point
    current_idx = torch.randint(0, N, (1,), device=device).item()
    
    for _ in range(n_samples):
        sampled_indices.append(current_idx)
        
        # Update distances
        current_point = points[current_idx]
        dist_to_current = torch.norm(points - current_point, dim=1)
        distances = torch.minimum(distances, dist_to_current)
        
        # Next point = farthest from all sampled
        current_idx = torch.argmax(distances).item()
    
    return torch.tensor(sampled_indices, dtype=torch.long, device=device)
 
 
# ─────────────────────────────────────────────────────
# 4. POINT CLOUD AUGMENTATION
# ─────────────────────────────────────────────────────
def augment_pointcloud(pos: torch.Tensor, norm: torch.Tensor) -> tuple:
    """
    Enhanced augmentation for 3D point clouds
    
    Augmentations:
      1. Y-axis rotation (objects can face any direction)
      2. Isotropic scaling (different distances/sizes)
      3. Gaussian jitter (sensor noise)
    
    WHY:
      - Y-rotation: ModelNet objects have random yaw, gravity is fixed
      - Scaling: Objects appear at different distances
      - Jitter: Simulates sensor noise
    
    Reference: Qi et al. 2017 (PointNet), Section 5
    
    Args:
        pos: [N, 3] positions (any device)
        norm: [N, 3] normals (any device)
    
    Returns:
        aug_pos: [N, 3] on CPU
        aug_norm: [N, 3] on CPU
    """
    pos = pos.detach().cpu().clone()
    norm = norm.detach().cpu().clone()
    
    # 1. Y-axis rotation
    angle_y = np.random.rand() * 2 * np.pi
    cos_y, sin_y = np.cos(angle_y), np.sin(angle_y)
    
    Ry = torch.tensor([
        [ cos_y, 0, sin_y],
        [     0, 1,     0],
        [-sin_y, 0, cos_y]
    ], dtype=torch.float32)
    
    pos = pos @ Ry.T
    norm = norm @ Ry.T  # Rotate normals too!
    
    # 2. Isotropic scaling
    scale = 0.95 + np.random.rand() * 0.10  # [0.95, 1.05]
    pos = pos * scale
    # Don't scale normals (should stay unit vectors)
    
    # 3. Gaussian jitter
    noise = torch.randn_like(pos) * 0.01
    noise = torch.clamp(noise, -0.02, 0.02)
    pos = pos + noise
    
    return pos, norm
 
 
def apply_augmentation(batch_data):
    """
    Apply augmentation to batch (CPU processing)
    
    Args:
        batch_data: PyG Batch object
    
    Returns:
        Augmented batch
    """
    batch_data = batch_data.cpu()
    data_list = batch_data.to_data_list()
    
    for data in data_list:
        # Extract components
        pos = data.pos
        norm = data.norm if hasattr(data, 'norm') else data.x[:, 3:6]
        
        # Augment
        aug_pos, aug_norm = augment_pointcloud(pos, norm)
        
        # Update data
        data.pos = aug_pos
        if hasattr(data, 'norm'):
            data.norm = aug_norm
        
        # Rebuild x = [aug_pos | aug_norm | motif] or [aug_pos | aug_norm]
        if data.x.shape[1] == 7:
            # Has motif scores
            motif = data.x[:, 6:7]
            data.x = torch.cat([aug_pos, aug_norm, motif], dim=1)
        else:
            # No motif scores
            data.x = torch.cat([aug_pos, aug_norm], dim=1)
    
    return Batch.from_data_list(data_list)

In [9]:
if not MOTIFS_PRECOMPUTED:
    print("\nComputing triangle motif scores (one-time cost)...")
    train_list = add_motif_features(train_list, k=K_NEIGHBORS, desc='train')
    test_list = add_motif_features(test_list, k=K_NEIGHBORS, desc='test')
    
    # Save for next time
    print("Saving motif cache...")
    torch.save({
        'train': train_list,
        'test': test_list,
        'classes': CLASSES
    }, 'modelnet40_with_motifs.pt')
    print(f"✓ Saved to modelnet40_with_motifs.pt")
 
 
# ─────────────────────────────────────────────────────
# 6. PRECOMPUTE K-NN GRAPHS
# ─────────────────────────────────────────────────────
print("\n🚀 Precomputing k-NN graphs (one-time cost)...")
 
def add_cached_graph(data, k=20):
    """Add precomputed edge_index to data object"""
    edge_index = knn_graph(data.pos, k=k, loop=False)
    data.edge_index = edge_index
    return data
 
train_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(train_list, desc='Train graphs')]
test_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(test_list, desc='Test graphs')]
 
print("✅ Graphs cached!")
 
 
# ─────────────────────────────────────────────────────
# 7. OPTIMIZED DATALOADERS
# ─────────────────────────────────────────────────────
train_loader = DataLoader(
    train_list,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)
 
test_loader = DataLoader(
    test_list,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)
 
print(f"\nDataLoaders ready:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")
 
 
# ─────────────────────────────────────────────────────
# 8. OPTIMIZED DGCNN MODEL
# ─────────────────────────────────────────────────────
from torch_geometric.nn import EdgeConv as PyGEdgeConv
 
class DGCNN(nn.Module):
    """
    Optimized DGCNN baseline
    
    Architecture:
      Input (6 features) → EdgeConv1 → EdgeConv2 → EdgeConv3 → EdgeConv4
      → Multi-scale aggregation → Global max pool → Classifier
    
    Optimizations:
      - Uses cached graphs (precomputed)
      - Reuses same graph for all layers (3× faster)
      - Mixed precision compatible
    
    Expected: 90-92% test OA on ModelNet40
    """
    def __init__(self, in_channels=6, num_classes=40, k=20, dropout=0.5):
        super().__init__()
        self.k = k
        
        # Input transformation
        self.input_transform = nn.Sequential(
            nn.Linear(in_channels, 64, bias=False),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(negative_slope=0.2)
        )
        
        # EdgeConv layers
        self.conv1 = PyGEdgeConv(
            nn.Sequential(
                nn.Linear(64 * 2, 64, bias=False),
                nn.BatchNorm1d(64),
                nn.LeakyReLU(0.2),
                nn.Linear(64, 64, bias=False),
                nn.BatchNorm1d(64),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        
        self.conv2 = PyGEdgeConv(
            nn.Sequential(
                nn.Linear(64 * 2, 64, bias=False),
                nn.BatchNorm1d(64),
                nn.LeakyReLU(0.2),
                nn.Linear(64, 64, bias=False),
                nn.BatchNorm1d(64),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        
        self.conv3 = PyGEdgeConv(
            nn.Sequential(
                nn.Linear(64 * 2, 128, bias=False),
                nn.BatchNorm1d(128),
                nn.LeakyReLU(0.2),
                nn.Linear(128, 128, bias=False),
                nn.BatchNorm1d(128),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        
        self.conv4 = PyGEdgeConv(
            nn.Sequential(
                nn.Linear(128 * 2, 256, bias=False),
                nn.BatchNorm1d(256),
                nn.LeakyReLU(0.2),
                nn.Linear(256, 256, bias=False),
                nn.BatchNorm1d(256),
                nn.LeakyReLU(0.2)
            ),
            aggr='max'
        )
        
        # Global feature aggregation
        self.global_mlp = nn.Sequential(
            nn.Linear(64 + 64 + 128 + 256, 1024, bias=False),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2)
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512, bias=False),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, data):
        x = data.x[:, :6]  # Use pos + norm only
        batch = data.batch
        
        # Use cached edge_index if available
        if hasattr(data, 'edge_index'):
            edge_index = data.edge_index
        else:
            edge_index = knn_graph(data.pos, k=self.k, batch=batch, loop=False)
        
        # Input transform
        x = self.input_transform(x)
        
        # EdgeConv layers (reuse same graph)
        x1 = self.conv1(x, edge_index)
        x2 = self.conv2(x1, edge_index)
        x3 = self.conv3(x2, edge_index)
        x4 = self.conv4(x3, edge_index)
        
        # Multi-scale aggregation
        x_cat = torch.cat([x1, x2, x3, x4], dim=1)
        x_global = self.global_mlp(x_cat)
        
        # Global pooling
        x_global = global_max_pool(x_global, batch)
        
        return self.classifier(x_global)
 


Computing triangle motif scores (one-time cost)...


Motif scores train:   0%|          | 0/9843 [00:00<?, ?it/s]

Motif scores test:   0%|          | 0/2468 [00:00<?, ?it/s]

Saving motif cache...
✓ Saved to modelnet40_with_motifs.pt

🚀 Precomputing k-NN graphs (one-time cost)...


Train graphs:   0%|          | 0/9843 [00:00<?, ?it/s]

Test graphs:   0%|          | 0/2468 [00:00<?, ?it/s]

✅ Graphs cached!

DataLoaders ready:
  Train batches: 821
  Test batches: 206


In [ ]:
# ─────────────────────────────────────────────────────
# 7.  PREPROCESSING — add motif features once, then save
# ─────────────────────────────────────────────────────
MOTIF_CACHE = '/kaggle/working/modelnet40_motif.pt'

if os.path.exists(MOTIF_CACHE):
    print("Loading motif-augmented cache …")
    motif_cache = torch.load(MOTIF_CACHE, map_location='cpu',weights_only = False)
    train_list  = motif_cache['train']
    test_list   = motif_cache['test']
else:
    print("Computing triangle motif scores (one-time cost) …")
    train_list = add_motif_features(train_list, k=K_NEIGHBORS, desc='train')
    test_list  = add_motif_features(test_list,  k=K_NEIGHBORS, desc='test')
    torch.save({'train': train_list, 'test': test_list,
                'classes': CLASSES}, MOTIF_CACHE)
    motif_size = os.path.getsize(MOTIF_CACHE) / 1e6
    print(f"Saved motif cache: {motif_size:.0f} MB")

# Verify feature shape
sample = train_list[0]
print(f"Node feature shape: {sample.x.shape}  ← should be [1024, 7]")
print(f"  col 0-2 : xyz  |  col 3-5 : normals  |  col 6 : triangle count")


# ─────────────────────────────────────────────────────
# 8.  TRAINING LOOP
# ─────────────────────────────────────────────────────
def apply_augmentation(batch_data):
    """
    Apply augmentation to batch - works on CPU
    """
    # Convert to list and work on CPU
    
    data_list = batch_data.cpu().to_data_list()  # Force CPU
    
    for data in data_list:
        # Augment position (on CPU)
        aug_pos, aug_norm = augment_pointcloud(data.pos, data.norm)
        motif = data.x[:, 6:7]
        data.pos = aug_pos
        data.norm = aug_norm
        
        # Rebuild x = [aug_pos | norm | triangle_count]
        data.x = torch.cat([aug_pos, aug_norm, motif], dim=1)
    
    return Batch.from_data_list(data_list)


# We need Batch for apply_augmentation
from torch_geometric.data import Batch

# num_workers=0 — Kaggle notebooks fork new processes inside an already
# forked process which breaks Python's multiprocessing assertions.
# num_workers=0 runs data loading in the main process, slightly slower
# but stable. On P100 the GPU is the bottleneck anyway, not data loading.
def add_cached_graph(data, k=20):
    """
    Add precomputed edge_index to data object
    This is done ONCE offline, saves 2-3× during training
    """
    edge_index = knn_graph(data.pos, k=k, loop=False)
    data.edge_index = edge_index
    return data

train_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(train_list, desc='Train graphs')]
test_list = [add_cached_graph(d, k=K_NEIGHBORS) for d in tqdm(test_list, desc='Test graphs')]

train_loader = DataLoader(
    train_list,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,           # Parallel data loading (4× speedup)
    pin_memory=True,         # Faster CPU→GPU transfer
    persistent_workers=True  # Keep workers alive between epochs
)
 
test_loader = DataLoader(
    test_list,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

model     = DGCNN(in_channels=6, num_classes=NUM_CLASSES, k=K_NEIGHBORS, dropout=0.5).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters : {total_params:,}")
print(f"Device           : {DEVICE}")
print(f"Epochs           : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}")


def train_epoch(model, loader, optimizer, scaler, device):
    """
    Training with mixed precision (FP16)
    Speedup: 1.5-2×
    """
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch in tqdm(loader, desc='  train', leave=False):
        batch = batch.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
        
        # Mixed precision forward pass
        with autocast():
            out = model(batch)
            loss = F.cross_entropy(out, batch.y.squeeze())
        
        # Backward with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item() * batch.num_graphs
        correct += out.argmax(1).eq(batch.y.squeeze()).sum().item()
        total += batch.num_graphs
    
    return total_loss / total, correct / total
 
 
@torch.no_grad()
def test_epoch(model, loader, device):
    """Evaluation with mixed precision"""
    model.eval()
    all_preds = []
    all_labels = []
    
    for batch in tqdm(loader, desc='  test ', leave=False):
        batch = batch.to(device, non_blocking=True)
        
        # Mixed precision inference
        with autocast():
            preds = model(batch).argmax(1).cpu().numpy()
        
        labels = batch.y.squeeze().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    return {
        'OA': accuracy_score(all_labels, all_preds) * 100,
        'mAcc': balanced_accuracy_score(all_labels, all_preds) * 100,
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0) * 100,
    }
 
 
# ─────────────────────────────────────────────────────
# 6. MAIN TRAINING SCRIPT
# ─────────────────────────────────────────────────────
 
print("\n" + "="*80)
print("TRAINING OPTIMIZED DGCNN")
print("="*80)
 
# Model
model = DGCNN(
    in_channels=6,
    num_classes=40,
    k=K_NEIGHBORS,
    dropout=0.5
).to(DEVICE)
 
print(f"\nModel: DGCNNOptimized")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Expected OA: 90-92%")
print(f"Expected time: ~40 minutes (vs 12+ hours baseline)")
 
# Optimizer & Scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
 
# Mixed precision scaler
scaler = GradScaler()
 
# Training history
best_oa = 0.0
history = {
    'epoch': [],
    'train_loss': [],
    'train_acc': [],
    'test_oa': [],
    'test_macc': [],
    'test_f1': []
}
 
# ─────────────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────────────
 
print(f"\nTraining from epoch 1 to {EPOCHS}...")
 
for epoch in trange(1, EPOCHS + 1, desc='Epochs'):
    # Train
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scaler, DEVICE)
    
    # Test
    metrics = test_epoch(model, test_loader, DEVICE)
    
    # Step scheduler
    scheduler.step()
    
    # Track best model
    if metrics['OA'] > best_oa:
        best_oa = metrics['OA']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_oa': best_oa,
        }, 'best_dgcnn_optimized.pt')
    
    # Record history
    history['epoch'].append(epoch)
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['test_oa'].append(metrics['OA'])
    history['test_macc'].append(metrics['mAcc'])
    history['test_f1'].append(metrics['macro_f1'])
    
    # Print every 5 epochs
    if epoch % 5 == 0:
        print(f"\n  Epoch {epoch:3d} | "
              f"loss {tr_loss:.4f} | "
              f"trAcc {tr_acc*100:.1f}% | "
              f"OA {metrics['OA']:.1f}% | "
              f"mAcc {metrics['mAcc']:.1f}% | "
              f"F1 {metrics['macro_f1']:.1f}%")
 
print("\n" + "="*80)
print(f"TRAINING COMPLETE")
print(f"Best OA: {best_oa:.2f}%")
print("="*80)
 
# ─────────────────────────────────────────────────────
# COMPARISON WITH BASELINE
# ─────────────────────────────────────────────────────
 
print("\nBaseline comparisons:")
print(f"  PointNet           → OA 89.2%  mAcc 86.2%")
print(f"  DGCNN (paper)      → OA 92.9%  mAcc 90.2%")
print(f"  Your DGCNN         → OA {best_oa:.1f}%  (expected 90-92%)")
print(f"  Target with pruning → OA 88-90% with 2× speedup")

Loading motif-augmented cache …
Node feature shape: torch.Size([1024, 8])  ← should be [1024, 7]
  col 0-2 : xyz  |  col 3-5 : normals  |  col 6 : triangle count


Train graphs:   0%|          | 0/9843 [00:00<?, ?it/s]

Test graphs:   0%|          | 0/2468 [00:00<?, ?it/s]


Model parameters : 1,384,744
Device           : cuda
Epochs           : 250  |  Batch: 12  |  LR: 0.001

TRAINING OPTIMIZED DGCNN

Model: DGCNNOptimized
Parameters: 1,384,744
Batch size: 12
Epochs: 250
Expected OA: 90-92%
Expected time: ~40 minutes (vs 12+ hours baseline)

Training from epoch 1 to 250...


/tmp/ipykernel_55/762203661.py:185: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


...

  test :   0%|          | 0/206 [00:00<?, ?it/s]

/tmp/ipykernel_55/762203661.py:140: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



  Epoch   5 | loss 1.1075 | trAcc 67.4% | OA 69.7% | mAcc 61.0% | F1 58.9%


  train:   0%|          | 0/821 [00:00<?, ?it/s]

/tmp/ipykernel_55/762203661.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
